# ERP Pipeline — All Participants, All 14 Channels (P3b Task)

**1. Settings**
All parameters in one place: paths, channel names, event codes (target=111, non-target=222), epoch window (-200 to 800 ms), baseline (-200 to 0 ms), P3b window (300-600 ms), and inclusion criteria (epoch retention).

**2. Helper functions**
- `set_channel_types` - assign EEG and stim channel labels
- `extract_events_from_marker` - extract event timestamps from the continuous marker channel
- `select_p3b_stim_events` - isolate P3b task events using task boundary markers (code=5) and verify counts (target=40, non-target=160)
- `mean_amplitude` - mean amplitude across a time window and channel set
- `load_behavioural_metadata` - read per-trial correctness/RT from `events.tsv`, aligned to epoch order (`None` if the file has no behavioural columns)

**3. Participant loop**
For each participant:
- Load EDF, set channel types, apply standard 10-20 montage
- Extract and filter P3b stimulus events
- Apply 0.1-30 Hz band-pass (plus a redundant 50 Hz notch)
- Mark bad channels from the QC file **before** average referencing
- Apply average reference (bad channels excluded automatically; scalp topography is therefore reference-dependent)
- Epoch around each stimulus, apply baseline correction
- Attach behavioural metadata (`condition`, `response`, `correct`, `response_status`, `reaction_time`) to `epochs.metadata` from `events.tsv`, verified against the EDF event order — no trials are dropped for this, it's metadata only
- Clean epochs with AutoReject (no separate ICA/EOG step: ocular artefacts are handled by AutoReject, and the posterior P3b ROI is largely unaffected by frontal blinks); metadata survives epoch dropping
- Check inclusion criteria -> `analysis_include`: retained_prop >= 50%, and >=20 target / >=80 non-target clean epochs (i.e. >=50% of trials retained in *each* condition, since the task always has 40 target + 160 non-target trials)
- Compute per-subject behavioural accuracy from all 200 trials (independent of AutoReject) as **diagnostic columns only** (overall + per-condition); no accuracy floor / exclusion flag is applied
- Manually flagged bad channels are excluded from the average reference but **not interpolated**; their (re-referenced) data remain in the all-channel output and are dropped at the analysis stage
- Save single-trial clean epochs (all 14 channels, with behavioural metadata) as **FIF** for flexible re-analysis (e.g. correct-only trials)
- Save full 14-channel evoked responses (all clean trials) as **FIF** (for topoplots) and **CSV** (long format, all channels)
- Extract mean P3b amplitude and waveform per channel - skip any channel flagged as excluded in the QC file

**4. Grand average & plots**
- Grand-average ERP waveforms and difference waves (Target - Non-target) for all 14 channels
- Grand-average topomaps from -200 to 600 ms (100 ms steps) for target, non-target, and difference

**5. Paths**
- Raw BIDS EDFs and `events.tsv` are read from `Notre_Dame_2026/Bids_conversion/` (untouched, read-only)
- Everything else — QC file, all outputs — lives under this review folder: `YBMAP_P3b_review/2_Preprocessing/ND_26/`

**6. Subject-exclusion flags — two independent sources, combine at analysis time**
- `analysis_include` (this pipeline) — epoch-retention / AutoReject thresholds (>=50% retained per condition)
- `exclude_subject` (`manual_channel_qc_nd26.csv`) — manual visual review of channel/signal quality
- e.g. `final_include = analysis_include & ~exclude_subject`
- (behavioural accuracy is recorded as a diagnostic column only, not an exclusion)

**7. Outputs** (in `2_Preprocessing/ND_26/preprocessing_output/`)
- `p3b_erp_summary_autoreject_nd_26_allch.csv` - per-channel P3b mean amplitudes
- `p3b_erp_waveforms_allch_autoreject_nd_26.csv` - all-channel waveforms
- `p3b_erp_processing_log_autoreject_nd_26_allch.csv` - QC / processing log
- `p3b_behavioural_qc_autoreject_nd_26.csv` - per-subject behavioural accuracy QC
- `evoked_fif/` - per-participant evoked (FIF, all clean trials) + grand-average FIFs
- `epochs_fif/` - per-participant single-trial clean epochs (FIF, with behavioural metadata)

In [ ]:
from pathlib import Path

import mne
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from autoreject import AutoReject

In [ ]:
Base = Path("/Users/miftahfaizah/Library/CloudStorage/OneDrive-UniversityofLeeds/PHD JOURNEY/YBMAP/CN_DATASET/Notre_Dame_2026")

REVIEW = Path("/Users/miftahfaizah/Library/CloudStorage/OneDrive-UniversityofLeeds/PHD JOURNEY/YBMAP/CN_DATASET/CN_analysis/YBMAP_P3b_review/2_Preprocessing/ND_26")

bids_root  = Base / "Bids_conversion"            # raw BIDS EDFs live in Notre_Dame_2026
output_dir = REVIEW / "preprocessing_output"     # outputs feed the YBMAP_P3b_review analysis
evoked_dir = output_dir / "evoked_fif"
epochs_dir = output_dir / "epochs_fif"           # single-trial epochs w/ behavioural metadata

output_dir.mkdir(parents=True, exist_ok=True)
evoked_dir.mkdir(parents=True, exist_ok=True)
epochs_dir.mkdir(parents=True, exist_ok=True)

task = "p3b"

eeg_channels = [
    "AF3", "F7", "F3", "FC5", "T7", "P7", "O1",
    "O2", "P8", "T8", "FC6", "F4", "F8", "AF4"
]

# All 14 channels used as regions — mean P3b amplitude extracted per channel
regions = {ch: [ch] for ch in eeg_channels}

# QC exclude flags only defined for occipital channels
region_exclude_col = {
    "O1": "exclude_O1",
    "O2": "exclude_O2",
}

event_id = {
    "target": 111,
    "non_target": 222,
}

p3b_window = (0.3, 0.6)

tmin     = -0.2
tmax     =  0.8
baseline = (None, 0)

min_retained_prop    = 0.50
min_target_epochs    = 20
min_nontarget_epochs = 80

random_state = 42

qc_df = pd.read_csv(REVIEW / "manual_channel_qc_nd26.csv")
qc_df["bad_channels"] = qc_df["bad_channels"].fillna("")

In [ ]:
def set_channel_types(raw):
    channel_types = {}
    for ch in raw.ch_names:
        if ch in eeg_channels:
            channel_types[ch] = "eeg"
        elif ch == "MarkerValueInt":
            channel_types[ch] = "stim"
        else:
            channel_types[ch] = "misc"
    raw.set_channel_types(channel_types)
    return raw


def extract_events_from_marker(raw, marker_channel="MarkerValueInt", marker_scale=1e6, min_gap_s=0.05):
    marker_data   = raw.copy().pick(marker_channel).get_data()[0]
    marker_scaled = np.round(marker_data * marker_scale).astype(int)

    sfreq       = raw.info["sfreq"]
    nonzero_idx = np.where(marker_scaled != 0)[0]

    event_samples = []
    event_codes   = []

    for idx in nonzero_idx:
        if len(event_samples) == 0:
            event_samples.append(idx)
            event_codes.append(marker_scaled[idx])
        else:
            previous_idx  = event_samples[-1]
            previous_code = event_codes[-1]
            is_new_code    = marker_scaled[idx] != previous_code
            is_far_enough  = (idx - previous_idx) / sfreq > min_gap_s
            if is_new_code or is_far_enough:
                event_samples.append(idx)
                event_codes.append(marker_scaled[idx])

    return np.column_stack([
        event_samples,
        np.zeros(len(event_samples), dtype=int),
        event_codes
    ]).astype(int)


def select_p3b_stim_events(events_all, expected_target=40, expected_non_target=160):
    p3b_bounds = events_all[events_all[:, 2] == 5]

    if len(p3b_bounds) < 2:
        raise ValueError(f"Expected at least two P3b task markers coded 5, found {len(p3b_bounds)}")

    p3b_start_sample = p3b_bounds[0, 0]
    p3b_end_sample   = p3b_bounds[1, 0]

    events = events_all[
        (events_all[:, 0] > p3b_start_sample)
        & (events_all[:, 0] < p3b_end_sample)
        & (np.isin(events_all[:, 2], [111, 222]))
    ].copy()

    counts      = pd.Series(events[:, 2]).value_counts().to_dict()
    n_target     = counts.get(111, 0)
    n_non_target = counts.get(222, 0)

    if n_target != expected_target or n_non_target != expected_non_target:
        raise ValueError(
            "Unexpected P3b stimulus counts after task-boundary filtering: "
            f"111={n_target}, 222={n_non_target}, total={len(events)}"
        )

    return events, p3b_start_sample, p3b_end_sample


def mean_amplitude(epochs, condition, channels, time_window):
    data      = epochs[condition].copy().pick(channels).get_data()
    times     = epochs.times
    tmin, tmax = time_window
    time_mask = (times >= tmin) & (times <= tmax)
    return data[:, :, time_mask].mean(axis=(1, 2)) * 1e6


def _norm_correct(v):
    """Normalise response_correct to True / False / pd.NA regardless of whether
    the source column parsed as bool or as the strings 'True'/'False'/'invalid'."""
    if isinstance(v, bool):
        return v
    if pd.isna(v):
        return pd.NA
    v = str(v).strip().lower()
    return True if v == "true" else False if v == "false" else pd.NA  # 'invalid' -> NA


def _response_status(v):
    """correct / incorrect / invalid (multi-press) / omission (no response)."""
    if pd.isna(v):
        return "omission"
    if isinstance(v, bool):
        return "correct" if v else "incorrect"
    v = str(v).strip().lower()
    return {"true": "correct", "false": "incorrect"}.get(v, "invalid")


def load_behavioural_metadata(events_file, expected_target, expected_non_target):
    """Trial-level behavioural metadata for the P3b stimulus events, in the same
    chronological order as the EDF-marker-derived epochs. Returns None for
    EDF-only sessions where events.tsv has no behavioural columns."""
    df = pd.read_csv(events_file, sep="\t")
    if "response_correct" not in df.columns:
        return None

    stim = (
        df[df["trial_type"].isin(["target", "non-target"])]
        .sort_values("onset")
        .reset_index(drop=True)
    )
    if len(stim) != expected_target + expected_non_target:
        raise ValueError(
            f"events.tsv stimulus count ({len(stim)}) != expected "
            f"({expected_target + expected_non_target}) in {events_file.name}"
        )

    return pd.DataFrame({
        "condition":       stim["trial_type"].map({"target": "target", "non-target": "non_target"}),
        "response":        stim["response_value"],
        "correct":         stim["response_correct"].map(_norm_correct),
        "response_status": stim["response_correct"].map(_response_status),
        "reaction_time":   pd.to_numeric(stim["response_rt"], errors="coerce"),
    })

In [ ]:
summary_rows     = []
log_rows         = []
waveform_rows    = []  # occipital ROI only — for ERP waveform plots
allch_waveform_rows = []  # all 14 channels — for topoplots
reject_log_rows  = []  # per-epoch x channel AutoReject detail
behavioural_rows = []  # per-subject behavioural accuracy QC
summary_rows_correct = []  # per-subject x channel P3b summary, CORRECT trials only

subjects = sorted([
    path.name
    for path in bids_root.glob("sub-*")
    if path.is_dir()
])

for subject in subjects:
    eeg_dir = bids_root / subject / "eeg"

    eeg_file    = eeg_dir / f"{subject}_task-{task}_eeg.edf"
    events_file = eeg_dir / f"{subject}_task-{task}_events.tsv"

    if not eeg_file.exists():
        log_rows.append({"subject": subject, "status": "skipped", "reason": "missing EDF file"})
        continue

    if not events_file.exists():
        log_rows.append({"subject": subject, "status": "skipped", "reason": "missing events.tsv file"})
        continue

    try:
        raw = mne.io.read_raw_edf(eeg_file, preload=True, verbose=False)
        raw = set_channel_types(raw)

        montage = mne.channels.make_standard_montage("standard_1020")
        raw.set_montage(montage, match_case=False, on_missing="ignore")

        events_all = extract_events_from_marker(raw)
        events, p3b_start_sample, p3b_end_sample = select_p3b_stim_events(events_all)

        raw_filt = raw.copy()
        # 0.1-30 Hz band-pass for P3b. The 50 Hz notch is redundant given the
        # 30 Hz low-pass but kept for explicitness. No ICA/EOG correction: ocular
        # artefacts are handled by AutoReject; the posterior P3b ROI is largely
        # unaffected by frontal blinks.
        raw_filt.notch_filter(freqs=50, picks="eeg", verbose=False)
        raw_filt.filter(l_freq=0.1, h_freq=30, picks="eeg", verbose=False)

        # Load bad channels from manual QC file before average reference
        qc_row = qc_df[qc_df["subject"] == subject]
        if not qc_row.empty and qc_row.iloc[0]["bad_channels"] != "":
            bad_chs = [ch.strip() for ch in qc_row.iloc[0]["bad_channels"].split(";")]
        else:
            bad_chs = []
        raw_filt.info["bads"] = bad_chs

        # Average reference (14 ch) — bad channels excluded automatically.
        # Note: scalp topography is therefore reference-dependent.
        raw_ref = raw_filt.copy().set_eeg_reference(
            ref_channels="average",
            projection=False,
            verbose=False
        )

        epochs = mne.Epochs(
            raw_ref,
            events,
            event_id=event_id,
            tmin=tmin,
            tmax=tmax,
            baseline=baseline,
            picks="eeg",
            preload=True,
            reject=None,
            verbose=False
        )

        # --- Behavioural metadata from events.tsv (target=40, non_target=160) ---
        # Attached before AutoReject so it survives epoch dropping/subsetting.
        # None for EDF-only sessions (no behavioural columns in events.tsv).
        behavioural = load_behavioural_metadata(events_file, 40, 160)

        if behavioural is not None:
            epoch_conditions = pd.Series(events[:, 2]).map({111: "target", 222: "non_target"}).reset_index(drop=True)
            if not (behavioural["condition"] == epoch_conditions).all():
                raise ValueError(f"Behavioural/EDF event order mismatch for {subject}")

        epochs.metadata = behavioural

        # --- Behavioural accuracy (descriptive, from all 200 trials, independent of AutoReject) ---
        # Per-subject overall and per-condition accuracy are stored as diagnostic
        # columns only. No accuracy floor / exclusion flag is applied.
        if behavioural is not None:
            is_target     = behavioural["condition"] == "target"
            is_nontarget  = behavioural["condition"] == "non_target"
            is_correct    = behavioural["correct"] == True

            n_target_trials    = int(is_target.sum())
            n_target_correct   = int((is_target & is_correct).sum())
            n_nontarget_trials = int(is_nontarget.sum())
            n_nontarget_correct = int((is_nontarget & is_correct).sum())

            target_accuracy    = n_target_correct / n_target_trials if n_target_trials else np.nan
            nontarget_accuracy = n_nontarget_correct / n_nontarget_trials if n_nontarget_trials else np.nan
            overall_accuracy   = (n_target_correct + n_nontarget_correct) / (n_target_trials + n_nontarget_trials)

            behavioural_rows.append({
                "subject":                     subject,
                "behaviour_available":         True,
                "n_target_trials":             n_target_trials,
                "n_target_correct":            n_target_correct,
                "target_accuracy":             target_accuracy,
                "n_nontarget_trials":          n_nontarget_trials,
                "n_nontarget_correct":         n_nontarget_correct,
                "nontarget_accuracy":          nontarget_accuracy,
                "overall_accuracy":            overall_accuracy,
                "n_invalid":                   int((behavioural["response_status"] == "invalid").sum()),
                "n_omission":                  int((behavioural["response_status"] == "omission").sum()),
            })
        else:
            behavioural_rows.append({
                "subject":                     subject,
                "behaviour_available":         False,
                "n_target_trials":             np.nan,
                "n_target_correct":            np.nan,
                "target_accuracy":             np.nan,
                "n_nontarget_trials":          np.nan,
                "n_nontarget_correct":         np.nan,
                "nontarget_accuracy":          np.nan,
                "overall_accuracy":            np.nan,
                "n_invalid":                   np.nan,
                "n_omission":                  np.nan,
            })

        ar = AutoReject(
            n_interpolate=[1, 2, 4],
            consensus=np.linspace(0.2, 0.8, 4),
            random_state=random_state,
            verbose=False
        )

        epochs_clean, reject_log = ar.fit_transform(epochs, return_log=True)

        # Single-trial epochs (all 14 channels, both conditions, w/ behavioural
        # metadata) — lets the analysis stage select correct-only trials or any
        # other metadata-based contrast without re-running preprocessing.
        epochs_clean.save(epochs_dir / f"{subject}-epo.fif", overwrite=True)

        n_original    = len(epochs)
        n_clean       = len(epochs_clean)
        n_dropped     = int(reject_log.bad_epochs.sum())
        retained_prop = n_clean / n_original if n_original > 0 else np.nan

        n_target_clean    = len(epochs_clean["target"])
        n_nontarget_clean = len(epochs_clean["non_target"])

        # --- Correct-trials-only epochs (parallel correct-only summary) ---
        # Uses behavioural metadata carried through AutoReject. None for EDF-only
        # sessions (no behaviour) -> correct-only summary skips those subjects.
        if epochs_clean.metadata is not None and "correct" in epochs_clean.metadata:
            correct_mask   = (epochs_clean.metadata["correct"] == True).fillna(False).to_numpy(dtype=bool)
            epochs_correct = epochs_clean[np.where(correct_mask)[0]] if correct_mask.any() else None
        else:
            epochs_correct = None

        if epochs_correct is not None:
            n_target_correct_clean    = len(epochs_correct["target"])
            n_nontarget_correct_clean = len(epochs_correct["non_target"])
            analysis_include_correct  = (
                retained_prop >= min_retained_prop
                and n_target_correct_clean >= min_target_epochs
                and n_nontarget_correct_clean >= min_nontarget_epochs
            )
        else:
            n_target_correct_clean = n_nontarget_correct_clean = 0
            analysis_include_correct = False

        analysis_include = (
            retained_prop >= min_retained_prop
            and n_target_clean >= min_target_epochs
            and n_nontarget_clean >= min_nontarget_epochs
        )

        # --- Per-epoch AutoReject reject log (epoch x channel detail) ---
        # status: good / interpolated / bad (rejected). Channels listed are those
        # AutoReject processed (manually flagged bad channels are excluded upstream).
        rl_labels   = reject_log.labels
        rl_chs      = list(reject_log.ch_names)
        rl_status   = {0: "good", 1: "bad", 2: "interpolated"}
        epoch_conds = pd.Series(epochs.events[:, 2]).map({111: "target", 222: "non_target"}).tolist()
        for ep_idx in range(rl_labels.shape[0]):
            ep_dropped = bool(reject_log.bad_epochs[ep_idx])
            for ch_idx, ch_name in enumerate(rl_chs):
                lbl = rl_labels[ep_idx, ch_idx]
                status = rl_status.get(int(lbl), "good") if lbl == lbl else "good"  # lbl==lbl is False for NaN
                reject_log_rows.append({
                    "subject":          subject,
                    "epoch_index":      ep_idx,
                    "condition":        epoch_conds[ep_idx],
                    "channel":          ch_name,
                    "status":           status,
                    "epoch_dropped":    ep_dropped,
                    "analysis_include": analysis_include,
                })

        # --- All-channel evoked: save as FIF and collect rows for CSV ---
        # Manually flagged bad channels are NOT interpolated (AutoReject ignores
        # info['bads']); their re-referenced data are written here and are dropped
        # later at the analysis stage.
        for condition in ["target", "non_target"]:
            evoked_all = epochs_clean[condition].average()  # all 14 channels

            # FIF — preserves MNE channel info and positions for direct topoplot use
            fif_path = evoked_dir / f"{subject}_{condition}_evoked-ave.fif"
            evoked_all.save(fif_path, overwrite=True)

            # CSV — long format, easy to open anywhere
            times_ms = evoked_all.times * 1000
            for ch_idx, ch_name in enumerate(evoked_all.ch_names):
                waveform_uv = evoked_all.data[ch_idx] * 1e6
                for time_ms, amplitude_uv in zip(times_ms, waveform_uv):
                    allch_waveform_rows.append({
                        "subject":          subject,
                        "condition":        condition,
                        "channel":          ch_name,
                        "time_ms":          time_ms,
                        "amplitude_uv":     amplitude_uv,
                        "analysis_include": analysis_include,
                    })

        # --- Occipital ROI: skip channels flagged as excluded in QC file ---
        for region_name, channels in regions.items():
            exclude_col = region_exclude_col.get(region_name)
            if not qc_row.empty and exclude_col is not None:
                region_excluded = bool(qc_row.iloc[0][exclude_col])
            else:
                region_excluded = False

            if region_excluded:
                continue

            target_amp    = mean_amplitude(epochs_clean, "target",     channels, p3b_window)
            nontarget_amp = mean_amplitude(epochs_clean, "non_target", channels, p3b_window)

            summary_rows.append({
                "subject":                  subject,
                "region":                   region_name,
                "channels":                 ", ".join(channels),
                "time_window_s":            f"{p3b_window[0]}-{p3b_window[1]}",
                "target_p3b_mean_uv":       target_amp.mean(),
                "non_target_p3b_mean_uv":   nontarget_amp.mean(),
                "target_minus_non_target_uv": target_amp.mean() - nontarget_amp.mean(),
                "n_target_epochs":          len(target_amp),
                "n_non_target_epochs":      len(nontarget_amp),
                "n_epochs_original":        n_original,
                "n_target_clean":           n_target_clean,
                "n_nontarget_clean":        n_nontarget_clean,
                "n_epochs_clean":           n_clean,
                "n_epochs_dropped":         n_dropped,
                "retained_prop":            retained_prop,
                "analysis_include":         analysis_include,
                "bad_channels":             ", ".join(bad_chs) if bad_chs else "",
                "p3b_start_sample":         p3b_start_sample,
                "p3b_end_sample":           p3b_end_sample,
                "n_edf_marker_events_all":  len(events_all),
                "n_p3b_stim_events_used":   len(events),
                "cleaning_method":          "autoreject",
            })

            # Correct-only counterpart (identical schema; amplitudes/counts from
            # correctly-responded trials only). analysis_include here is the
            # correct-only inclusion (target/non-target thresholds on correct trials).
            if epochs_correct is not None and n_target_correct_clean > 0 and n_nontarget_correct_clean > 0:
                tac = mean_amplitude(epochs_correct, "target",     channels, p3b_window)
                nac = mean_amplitude(epochs_correct, "non_target", channels, p3b_window)
                summary_rows_correct.append({
                    "subject":                  subject,
                    "region":                   region_name,
                    "channels":                 ", ".join(channels),
                    "time_window_s":            f"{p3b_window[0]}-{p3b_window[1]}",
                    "target_p3b_mean_uv":       tac.mean(),
                    "non_target_p3b_mean_uv":   nac.mean(),
                    "target_minus_non_target_uv": tac.mean() - nac.mean(),
                    "n_target_epochs":          len(tac),
                    "n_non_target_epochs":      len(nac),
                    "n_epochs_original":        n_original,
                    "n_target_clean":           n_target_correct_clean,
                    "n_nontarget_clean":        n_nontarget_correct_clean,
                    "n_epochs_clean":           n_target_correct_clean + n_nontarget_correct_clean,
                    "n_epochs_dropped":         n_dropped,
                    "retained_prop":            retained_prop,
                    "analysis_include":         analysis_include_correct,
                    "bad_channels":             ", ".join(bad_chs) if bad_chs else "",
                    "p3b_start_sample":         p3b_start_sample,
                    "p3b_end_sample":           p3b_end_sample,
                    "n_edf_marker_events_all":  len(events_all),
                    "n_p3b_stim_events_used":   len(events),
                    "cleaning_method":          "autoreject_correct_only",
                })

            for condition in ["target", "non_target"]:
                evoked   = epochs_clean[condition].copy().pick(channels).average()
                waveform_uv = evoked.data.mean(axis=0) * 1e6
                times_ms    = evoked.times * 1000

                for time_ms, amplitude_uv in zip(times_ms, waveform_uv):
                    waveform_rows.append({
                        "subject":          subject,
                        "region":           region_name,
                        "condition":        condition,
                        "time_ms":          time_ms,
                        "amplitude_uv":     amplitude_uv,
                        "analysis_include": analysis_include,
                    })

        log_rows.append({
            "subject":          subject,
            "status":           "ok",
            "reason":           "",
            "bad_channels":     ", ".join(bad_chs) if bad_chs else "",
            "n_epochs_original": n_original,
            "n_epochs_clean":   n_clean,
            "n_target_clean":    n_target_clean,
            "n_nontarget_clean": n_nontarget_clean,
            "n_epochs_dropped": n_dropped,
            "retained_prop":    retained_prop,
            "analysis_include": analysis_include,
        })

    except Exception as e:
        log_rows.append({
            "subject": subject,
            "status":  "error",
            "reason":  str(e),
        })

p3b_all_subjects  = pd.DataFrame(summary_rows)
erp_waveforms_all = pd.DataFrame(waveform_rows)
allch_waveforms   = pd.DataFrame(allch_waveform_rows)
processing_log    = pd.DataFrame(log_rows)
reject_log_df     = pd.DataFrame(reject_log_rows)
behavioural_qc    = pd.DataFrame(behavioural_rows)
p3b_all_subjects_correct = pd.DataFrame(summary_rows_correct)

p3b_all_subjects

In [ ]:
# Recovery load — ONLY needed if you restarted the kernel and SKIPPED the main
# participant loop above (e.g. to re-plot from saved CSVs without re-running EEG).
# Guarded so a full "Run All" does NOT overwrite the freshly-computed dataframes
# with the previously-saved (possibly stale) CSVs.
if "behavioural_qc" not in globals():
    p3b_all_subjects  = pd.read_csv(output_dir / "p3b_erp_summary_autoreject_nd_26_allch.csv")
    erp_waveforms_all = pd.read_csv(output_dir / "p3b_erp_waveforms_roi_autoreject_nd_26.csv")
    allch_waveforms   = pd.read_csv(output_dir / "p3b_erp_waveforms_allch_autoreject_nd_26.csv")
    processing_log    = pd.read_csv(output_dir / "p3b_erp_processing_log_autoreject_nd_26_allch.csv")
    reject_log_df     = pd.read_csv(output_dir / "p3b_reject_log_autoreject_nd_26_allch.csv")
    behavioural_qc    = pd.read_csv(output_dir / "p3b_behavioural_qc_autoreject_nd_26.csv")
    print("Recovery: loaded dataframes from saved CSVs (main loop was skipped).")
else:
    print("Main-loop dataframes already in memory — recovery load skipped.")

In [ ]:
processing_log

In [ ]:
erp_waveforms_included = erp_waveforms_all[
    erp_waveforms_all["analysis_include"] == True
].copy()

n_included = erp_waveforms_included["subject"].nunique()

grand_average = (
    erp_waveforms_included
    .groupby(["region", "condition", "time_ms"], as_index=False)
    ["amplitude_uv"]
    .mean()
)

grand_average.head()

In [ ]:
n_channels = len(eeg_channels)
ncols = 4
nrows = -(-n_channels // ncols)  # ceiling division

fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 3), sharey=True)
axes_flat = axes.flatten()

for ax, ch_name in zip(axes_flat, eeg_channels):
    ch_data = grand_average[grand_average["region"] == ch_name]

    target     = ch_data[ch_data["condition"] == "target"].sort_values("time_ms")
    non_target = ch_data[ch_data["condition"] == "non_target"].sort_values("time_ms")

    ax.plot(target["time_ms"],     target["amplitude_uv"],     label="Target",     color="tab:blue")
    ax.plot(non_target["time_ms"], non_target["amplitude_uv"], label="Non-target", color="tab:orange")

    ax.axvline(0, color="black", linestyle="--", linewidth=0.8)
    ax.axhline(0, color="black", linewidth=0.6)
    ax.axvspan(300, 600, color="grey", alpha=0.2)

    ax.set_title(ch_name)
    ax.set_xlabel("Time (ms)")

# Hide unused axes
for ax in axes_flat[n_channels:]:
    ax.set_visible(False)

axes_flat[0].set_ylabel("Amplitude (µV)")
axes_flat[ncols - 1].legend()

plt.suptitle(f"Grand Average ERP — All Channels (Included Participants, n={n_included})")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 3), sharey=True)
axes_flat = axes.flatten()

for ax, ch_name in zip(axes_flat, eeg_channels):
    ch_data = grand_average[grand_average["region"] == ch_name]

    target     = ch_data[ch_data["condition"] == "target"].sort_values("time_ms")
    non_target = ch_data[ch_data["condition"] == "non_target"].sort_values("time_ms")

    difference = target["amplitude_uv"].to_numpy() - non_target["amplitude_uv"].to_numpy()

    ax.plot(target["time_ms"], difference, color="tab:green")
    ax.axvline(0, color="black", linestyle="--", linewidth=0.8)
    ax.axhline(0, color="black", linewidth=0.6)
    ax.axvspan(300, 600, color="grey", alpha=0.2)

    ax.set_title(ch_name)
    ax.set_xlabel("Time (ms)")

for ax in axes_flat[n_channels:]:
    ax.set_visible(False)

axes_flat[0].set_ylabel("Amplitude difference (µV)")

plt.suptitle(f"Grand Average P3b Difference Wave — All Channels (Target − Non-target, n={n_included})")
plt.tight_layout()
plt.show()

## Strict Grand Average (Epoch + Manual QC Combined)

The plots above only apply `analysis_include` (epoch-retention QC). The two cells below
additionally drop subjects flagged by `exclude_subject` (manual channel/signal-quality QC),
using all 14 channels. Behavioural accuracy is a diagnostic column only and is not applied.

In [ ]:

# Strict subject set: epoch-retention QC AND manual channel/signal-quality QC.
# A behavioural-accuracy floor is NOT applied as an exclusion -- overall accuracy
# is kept only as a diagnostic column in p3b_behavioural_qc_*.csv.
# Strict = analysis_include & ~exclude_subject.
epoch_ok        = set(processing_log.loc[processing_log["analysis_include"] == True, "subject"])
manual_excluded = set(qc_df.loc[qc_df["exclude_subject"] == True, "subject"])

strict_included_subjects = epoch_ok - manual_excluded
print(
    f"{len(strict_included_subjects)} / {len(epoch_ok)} epoch-included subjects also pass "
    f"manual channel QC (behavioural-accuracy floor not applied)"
)

allch_waveforms_strict = allch_waveforms[
    allch_waveforms["subject"].isin(strict_included_subjects)
].copy()

grand_average_strict = (
    allch_waveforms_strict
    .groupby(["condition", "channel", "time_ms"], as_index=False)
    ["amplitude_uv"]
    .mean()
)

fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 3), sharey=True)
axes_flat = axes.flatten()

for ax, ch_name in zip(axes_flat, eeg_channels):
    ch_data = grand_average_strict[grand_average_strict["channel"] == ch_name]

    target     = ch_data[ch_data["condition"] == "target"].sort_values("time_ms")
    non_target = ch_data[ch_data["condition"] == "non_target"].sort_values("time_ms")

    ax.plot(target["time_ms"],     target["amplitude_uv"],     label="Target",     color="tab:blue")
    ax.plot(non_target["time_ms"], non_target["amplitude_uv"], label="Non-target", color="tab:orange")

    ax.axvline(0, color="black", linestyle="--", linewidth=0.8)
    ax.axhline(0, color="black", linewidth=0.6)
    ax.axvspan(300, 600, color="grey", alpha=0.2)

    ax.set_title(ch_name)
    ax.set_xlabel("Time (ms)")

for ax in axes_flat[n_channels:]:
    ax.set_visible(False)

axes_flat[0].set_ylabel("Amplitude (µV)")
axes_flat[ncols - 1].legend()

plt.suptitle(f"Grand Average ERP — All Channels (Strict QC, n={len(strict_included_subjects)})")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 3), sharey=True)
axes_flat = axes.flatten()

for ax, ch_name in zip(axes_flat, eeg_channels):
    ch_data = grand_average_strict[grand_average_strict["channel"] == ch_name]

    target     = ch_data[ch_data["condition"] == "target"].sort_values("time_ms")
    non_target = ch_data[ch_data["condition"] == "non_target"].sort_values("time_ms")

    difference = target["amplitude_uv"].to_numpy() - non_target["amplitude_uv"].to_numpy()

    ax.plot(target["time_ms"], difference, color="tab:green")
    ax.axvline(0, color="black", linestyle="--", linewidth=0.8)
    ax.axhline(0, color="black", linewidth=0.6)
    ax.axvspan(300, 600, color="grey", alpha=0.2)

    ax.set_title(ch_name)
    ax.set_xlabel("Time (ms)")

for ax in axes_flat[n_channels:]:
    ax.set_visible(False)

axes_flat[0].set_ylabel("Amplitude difference (µV)")

plt.suptitle(f"Grand Average P3b Difference Wave — All Channels (Strict QC, n={len(strict_included_subjects)})")
plt.tight_layout()
plt.show()

## Topographic Maps (Grand Average)

Scalp amplitude distribution plotted at 300, 400, 500, and 600 ms for target and non-target conditions.
Grand average is computed from per-participant FIF files (included participants only).
FIF format is used here because it preserves MNE channel position metadata, allowing topoplots to be generated directly.

In [ ]:
topomap_times = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]  # seconds

included_subjects = set(
    processing_log.loc[processing_log["analysis_include"] == True, "subject"]
)

# Load included evoked per condition
evoked_per_condition = {}
for condition in ["target", "non_target"]:
    evoked_list = []
    for fif_path in sorted(evoked_dir.glob(f"*_{condition}_evoked-ave.fif")):
        sub = fif_path.name.split(f"_{condition}")[0]
        if sub in included_subjects:
            evoked_list.append(mne.read_evokeds(fif_path, verbose=False)[0])

    if not evoked_list:
        print(f"No included evoked files found for condition: {condition}")
        continue

    grand_avg = mne.grand_average(evoked_list)
    evoked_per_condition[condition] = grand_avg

    grand_avg_path = output_dir / f"grand_average_{condition}_evoked-ave.fif"
    grand_avg.save(grand_avg_path, overwrite=True)
    print(f"Saved: {grand_avg_path}")

# --- Target ---
fig = evoked_per_condition["target"].plot_topomap(
    times=topomap_times, average=0.05, show=False
)
fig.suptitle("Grand Average Topomap — Target", y=1.02)
plt.show()

# --- Non-target ---
fig = evoked_per_condition["non_target"].plot_topomap(
    times=topomap_times, average=0.05, show=False
)
fig.suptitle("Grand Average Topomap — Non-target", y=1.02)
plt.show()

# --- Difference (Target − Non-target) ---
diff_evoked = evoked_per_condition["target"].copy()
diff_evoked.data = evoked_per_condition["target"].data - evoked_per_condition["non_target"].data

fig = diff_evoked.plot_topomap(
    times=topomap_times, average=0.05, show=False
)
fig.suptitle("Grand Average Topomap — Difference (Target − Non-target)", y=1.02)
plt.show()

In [ ]:
erp_out         = output_dir / "p3b_erp_summary_autoreject_nd_26_allch.csv"
erp_correct_out = output_dir / "p3b_erp_summary_correctonly_autoreject_nd_26_allch.csv"
allch_out       = output_dir / "p3b_erp_waveforms_allch_autoreject_nd_26.csv"
log_out         = output_dir / "p3b_erp_processing_log_autoreject_nd_26_allch.csv"
reject_out      = output_dir / "p3b_reject_log_autoreject_nd_26_allch.csv"
behavioural_out = output_dir / "p3b_behavioural_qc_autoreject_nd_26.csv"

# NOTE: p3b_erp_waveforms_roi_*.csv is no longer written. With `regions` defined
# as single channels it duplicated p3b_erp_waveforms_allch_*.csv exactly (the ROI
# rows were a channel subset, differing only by the column name). No active ND
# analysis reads it; use the all-channel waveform file instead.

p3b_all_subjects.to_csv(erp_out,                 index=False)
p3b_all_subjects_correct.to_csv(erp_correct_out, index=False)
allch_waveforms.to_csv(allch_out,                index=False)
processing_log.to_csv(log_out,                   index=False)
reject_log_df.to_csv(reject_out,                 index=False)
behavioural_qc.to_csv(behavioural_out,           index=False)

print("Saved:")
print(f"  {erp_out}")
print(f"  {erp_correct_out}")
print(f"  {allch_out}")
print(f"  {log_out}")
print(f"  {reject_out}")
print(f"  {behavioural_out}")
